### Data wrangling without loss

Inspect the “patients_lab_results_messy.csv” dataset

What is wrong with the dataset? Look for e.g. 
- Inconsistent labelling
- Inconsistent casing
- Missing values

Fix the issues without loosing any data! 

Discuss the reversibility of your steps and think about how you can make sure that you did not loose any data!


In [ ]:
import pandas as pd
import seaborn as sns

df = pd.read_csv("patients_results_messy.csv")
df

Clean up column names using string methods:

In [ ]:
df.columns = df.columns.str.lower().str.replace('[^a-z0-9]+', '_', regex=True).str.strip('_')
df

Split blood pressure values into two columns, using the string method split():

In [ ]:
df[['systolic', 'diastolic']] = df['blood_pressure_mmhg'].str.split('/', expand=True)
df

In [ ]:
df.systolic = df.systolic.apply(pd.to_numeric)
df.diastolic = df.diastolic.apply(pd.to_numeric)
df

Convert glucose values:

In [ ]:
df['glucose'] = pd.to_numeric(df['glucose'], errors="coerce") # avoids crashing if wrong types are parsed
df

Standardize gender:

In [ ]:
df['gender'] = df['gender'].str.upper().map({'MALE': 'M', 'FEMALE': 'F', 'M': 'M', 'F': 'F'})
df

Parse dates to standard format:

In [ ]:
# standardize visit_date
df['visit_date'] = (
    df['visit_date']
    .str.replace(r'[./]', '-', regex=True)  # unify separators
    .str.strip()
)

df['visit_date_clean'] = pd.to_datetime(df['visit_date'], format='mixed')
df

Further cleanup: patient_id!

In [ ]:
df.patient_id = df.patient_id.str.upper()
df

### Concatenation and merging of dataframes

Add the extended patient list and the patients info to your cleaned up dataframe. Inspect the shape of the dataframes after each operation.

In [ ]:
df_extension = pd.read_csv("patient_list_extended.csv")
df_2merge = pd.read_csv("patients_info.csv")

1) Concatenation. Try different options, e.g. `axis=`

In [ ]:
df_extended = pd.concat([df,df_extension], ignore_index=True)
df_extended.shape

2) Merging. Play with different modes (`how=`)

In [ ]:
df_complete = pd.merge(df_extended, df_2merge, on="patient_id", how="outer")
df_complete.shape

`join()` can be used to the same purpose, but requires an index to do so:

In [ ]:
df_complete2 = df_extended.join(df_2merge.set_index("patient_id"), on="patient_id", how="outer")
df_complete2.shape

Write to a csv.

In [ ]:
df_complete.to_csv("patients_list_complete.csv",index=False)

### Data wrangling with loss

Intentional loss / deletion of data always needs to be documented / commented. In this notebook this is only done in a very minimalistic version for brevity.

Assess the `NaN` in your combined dataframe.

Flag missing values, e.g. for age or glucose values.

Drop rows where a defined column, e.g. glucose, contains an `NaN`.

Assess duplicates. Assess partial duplicates using `subset=`. Delete if needed.

### Outlier filtering

Use the glucose as an example feature to practice outlier assessment. Check in a plot how the pattern is affected by the changes.

The classic: Innerquartile range (see Box plot)

Filter as to whatever lies within the IQR.

In [ ]:
# IQR
Q1 = df_complete.glucose.quantile(0.25)
Q3 = df_complete.glucose.quantile(0.75)
IQR = Q3-Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

print(Q1, Q3)

In [ ]:
filtered_df = df[(df_complete.glucose >= lower) & (df_complete.glucose <= upper)]
filtered_df.shape

In [ ]:
sns.histplot(filtered_df.glucose)

Clip the data at a certain quantile, using the ``clip`` method.

In [ ]:
lower = df_complete.chol_mg_dl.quantile(0.02)
upper = df_complete.chol_mg_dl.quantile(0.98)

clipped_df = df_complete.chol_mg_dl.clip(lower, upper)
clipped_df.head()


Other outlier assessment exist, e.g. z-scores. A point is considered an outlier if its z-score is lower than -3 or higher than 3 (i.e. is further away from the mean than three standard deviations)

In [ ]:
from scipy import stats
# zscore fails if NaNs are not removed, use a df where missing values for glucose are removed
df_dropna_glu = df_complete.dropna(subset=["glucose"])
filtered_df2 = df_dropna_glu[abs(stats.zscore(df_dropna_glu.glucose)) < 3] 
filtered_df2.shape